# IMERG 2025-2026: Download và crop Vietnam

Notebook này chỉ tải dữ liệu IMERG Final Run half-hourly (`GPM_3IMERGHH`, V07) cho giai đoạn 2025-2026.

File 2023-2024 được giữ nguyên và không được đọc ghi nối hoặc tải lại. Output của notebook là `imerg_vietnam_2025_2026.nc`.

> Cần tài khoản NASA Earthdata. Khoảng thời gian tải dùng mốc cuối loại trừ: `2025-01-01` đến trước `2027-01-01`.

In [ ]:
import sys
import subprocess
from pathlib import Path

if "google.colab" in sys.modules:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "xarray", "netCDF4", "h5py", "earthaccess",
        "pandas", "numpy",
    ])

import h5py
import numpy as np
import pandas as pd

try:
    from google.colab import drive

    drive.mount("/content/drive")
    IN_COLAB = True
    BASE_DIR = Path("/content/drive/MyDrive/Rainfall_Nowcasting/IMERG")
except ImportError:
    IN_COLAB = False
    BASE_DIR = Path.cwd().resolve()

# End is exclusive. Existing frames in VN_FILE are preserved and skipped.
START = "2025-01-01T00:00:00"
END = "2027-01-01T00:00:00"
VN_BBOX = (102.0, 8.0, 110.0, 24.5)
RUN_DOWNLOAD = True

RAW_DIR = BASE_DIR / "data" / "imerg_raw_2025_2026"
PROCESSED_DIR = BASE_DIR / "data" / "imerg_vietnam"
TEMP_RAW_DIR = Path("/content/imerg_batch_2025_2026") if IN_COLAB else BASE_DIR / "data" / "imerg_batch_2025_2026"
VN_FILE = PROCESSED_DIR / "imerg_vietnam_2025_2026.nc"
FAILED_LOG = PROCESSED_DIR / "failed_granules_2025_2026.json"

for directory in (RAW_DIR, PROCESSED_DIR, TEMP_RAW_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Running in Colab: {IN_COLAB}")
print(f"Output: {VN_FILE}")
print(f"Time range: {START} <= time < {END}")
print("Resume mode: existing frames in the output file will not be downloaded again.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Running in Colab: True
Output: /content/drive/MyDrive/Rainfall_Nowcasting/IMERG/data/imerg_vietnam/imerg_vietnam_2025_2026.nc
Time range: 2025-01-01T00:00:00 <= time < 2027-01-01T00:00:00
Resume mode: existing frames in the output file will not be downloaded again.


In [ ]:
import gc
import json
import re
import time

import earthaccess
from netCDF4 import Dataset, date2num, num2date

def granule_time(granule):
    value = granule["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"]
    return pd.Timestamp(value).tz_localize(None).to_pydatetime()

def existing_timestamps(output_path):
    if not output_path.exists():
        return set()
    with Dataset(output_path, "r") as output:
        time_var = output.variables["time"]
        dates = num2date(
            time_var[:],
            time_var.units,
            calendar=getattr(time_var, "calendar", "standard"),
            only_use_cftime_datetimes=False,
        )
    return {pd.Timestamp(date).tz_localize(None).to_pydatetime() for date in dates}

earthaccess.login(strategy="interactive", persist=True)
completed = existing_timestamps(VN_FILE)
resume_start = max(completed) + pd.Timedelta(minutes=30) if completed else pd.Timestamp(START).to_pydatetime()
search_start = max(resume_start, pd.Timestamp(START).to_pydatetime())

def search_missing_granules(start, end):
    found = []
    cursor = pd.Timestamp(start)
    final = pd.Timestamp(end)
    while cursor < final:
        chunk_end = min(cursor + pd.DateOffset(months=1), final)
        chunk = earthaccess.search_data(
            short_name="GPM_3IMERGHH",
            version="07",
            temporal=(cursor.isoformat(), chunk_end.isoformat()),
            bounding_box=VN_BBOX,
            count=-1,
        )
        found.extend(chunk)
        print(f"Search {cursor:%Y-%m-%d} -> {chunk_end:%Y-%m-%d}: {len(chunk):,} granules")
        cursor = chunk_end
    unique = {granule_time(item): item for item in found}
    return [unique[timestamp] for timestamp in sorted(unique)]

granules = search_missing_granules(search_start, END)
pending_granules = [item for item in granules if granule_time(item) not in completed]
print(f"Existing frames in {VN_FILE.name}: {len(completed):,}")
print(f"Resume search starts at: {search_start}")
if granules:
    print(f"Granules returned: {granule_time(granules[0])} -> {granule_time(granules[-1])}")
else:
    print("No granules were returned for the missing period.")
print(f"Pending granules to download: {len(pending_granules):,}")

BATCH_SIZE = 100
MAX_RETRIES = 3

def read_hdf5_dataset(group, names):
    found = []
    def visitor(name, value):
        if isinstance(value, h5py.Dataset) and name.split("/")[-1] in names:
            found.append(value)
    group.visititems(visitor)
    if not found:
        raise KeyError(f"Could not find any of {sorted(names)}")
    return found[0]

def crop_file_to_frame(file_path):
    with h5py.File(file_path, "r") as source:
        grid = source["Grid"] if "Grid" in source else source
        rainfall = np.asarray(read_hdf5_dataset(grid, {"precipitationCal", "precipitation"})[:], dtype="float32").squeeze()
        latitudes = np.asarray(read_hdf5_dataset(grid, {"lat", "latitude"})[:], dtype="float32").squeeze()
        longitudes = np.asarray(read_hdf5_dataset(grid, {"lon", "longitude"})[:], dtype="float32").squeeze()

    if rainfall.shape == (len(longitudes), len(latitudes)):
        rainfall = rainfall.T
    if rainfall.shape != (len(latitudes), len(longitudes)):
        raise ValueError(f"Cannot align rainfall shape {rainfall.shape} with lat/lon arrays")

    lat_order = np.argsort(latitudes)
    lon_order = np.argsort(longitudes)
    latitudes = latitudes[lat_order]
    longitudes = longitudes[lon_order]
    rainfall = rainfall[np.ix_(lat_order, lon_order)]
    lat_mask = (latitudes >= VN_BBOX[1]) & (latitudes <= VN_BBOX[3])
    lon_mask = (longitudes >= VN_BBOX[0]) & (longitudes <= VN_BBOX[2])
    if not lat_mask.any() or not lon_mask.any():
        raise ValueError("Vietnam bounding box does not overlap this granule")

    rainfall = rainfall[np.ix_(lat_mask, lon_mask)] * 0.5
    rainfall = np.where(np.isfinite(rainfall) & (rainfall >= 0), rainfall, np.nan)
    match = re.search(r"\.(\d{8})-S(\d{6})-", Path(file_path).name)
    if not match:
        raise ValueError(f"Cannot parse timestamp from {file_path}")
    timestamp = pd.to_datetime(match.group(1) + match.group(2), format="%Y%m%d%H%M%S")
    return timestamp.to_pydatetime(), latitudes[lat_mask], longitudes[lon_mask], rainfall

def append_frame(timestamp, latitudes, longitudes, rainfall):
    time_units = "hours since 1970-01-01 00:00:00"
    if not VN_FILE.exists():
        with Dataset(VN_FILE, "w", format="NETCDF4") as output:
            output.createDimension("time", None)
            output.createDimension("lat", len(latitudes))
            output.createDimension("lon", len(longitudes))
            time_var = output.createVariable("time", "f8", ("time",))
            output.createVariable("lat", "f4", ("lat",))[:] = latitudes
            output.createVariable("lon", "f4", ("lon",))[:] = longitudes
            rain_var = output.createVariable("rainfall", "f4", ("time", "lat", "lon"), zlib=True, complevel=4, fill_value=np.float32(np.nan), chunksizes=(1, len(latitudes), len(longitudes)))
            time_var.units = time_units
            time_var.calendar = "standard"
            rain_var.units = "mm/30 min"
            rain_var.long_name = "IMERG precipitation accumulation per 30 minutes"
            rain_var.source = "NASA GPM IMERG Final Run V07"
            output.title = "IMERG rainfall cropped to Vietnam, 2025-2026"
    with Dataset(VN_FILE, "a") as output:
        index = len(output.dimensions["time"])
        output.variables["time"][index] = date2num(timestamp, time_units, calendar="standard")
        output.variables["rainfall"][index, :, :] = rainfall

def process_downloaded_files(completed):
    for file_path in sorted(TEMP_RAW_DIR.glob("*.HDF5")):
        try:
            timestamp, latitudes, longitudes, rainfall = crop_file_to_frame(file_path)
            if timestamp not in completed:
                append_frame(timestamp, latitudes, longitudes, rainfall)
                completed.add(timestamp)
        except (OSError, KeyError, ValueError) as error:
            print(f"Could not process {file_path.name}: {error}")
        finally:
            file_path.unlink(missing_ok=True)
    gc.collect()
    return completed

def download_with_retry(items):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            earthaccess.download(items, str(TEMP_RAW_DIR))
            return True
        except Exception as error:
            print(f"Download attempt {attempt}/{MAX_RETRIES} failed: {error}")
            if attempt < MAX_RETRIES:
                time.sleep(5 * attempt)
    return False

if RUN_DOWNLOAD:
    failed = []
    for batch_start in range(0, len(pending_granules), BATCH_SIZE):
        batch = pending_granules[batch_start : batch_start + BATCH_SIZE]
        download_with_retry(batch)
        completed = process_downloaded_files(completed)
        for item in batch:
            expected = granule_time(item)
            if expected not in completed:
                download_with_retry([item])
                completed = process_downloaded_files(completed)
                if expected not in completed:
                    failed.append(str(item))
        print(
            f"Processed {min(batch_start + BATCH_SIZE, len(pending_granules)):,}/"
            f"{len(pending_granules):,} pending granules; "
            f"saved {len(completed):,} total frames"
        )
    if failed:
        FAILED_LOG.write_text(json.dumps(failed, indent=2), encoding="utf-8")
        print(f"Warning: {len(failed):,} granules failed; see {FAILED_LOG}")
    print(f"Saved 2025-2026 data to: {VN_FILE}")
else:
    print("RUN_DOWNLOAD=False: khong tai du lieu.")

Search 2025-10-01 -> 2025-11-01: 0 granules
Search 2025-11-01 -> 2025-12-01: 0 granules
Search 2025-12-01 -> 2026-01-01: 0 granules
Search 2026-01-01 -> 2026-02-01: 0 granules
Search 2026-02-01 -> 2026-03-01: 0 granules
Search 2026-03-01 -> 2026-04-01: 0 granules
Search 2026-04-01 -> 2026-05-01: 0 granules
Search 2026-05-01 -> 2026-06-01: 0 granules
Search 2026-06-01 -> 2026-07-01: 0 granules
Search 2026-07-01 -> 2026-08-01: 0 granules
Search 2026-08-01 -> 2026-09-01: 0 granules
Search 2026-09-01 -> 2026-10-01: 0 granules
Search 2026-10-01 -> 2026-11-01: 0 granules
Search 2026-11-01 -> 2026-12-01: 0 granules
Search 2026-12-01 -> 2027-01-01: 0 granules
Existing frames in imerg_vietnam_2025_2026.nc: 13,104
Resume search starts at: 2025-10-01 00:00:00
No granules were returned for the missing period.
Pending granules to download: 0
Saved 2025-2026 data to: /content/drive/MyDrive/Rainfall_Nowcasting/IMERG/data/imerg_vietnam/imerg_vietnam_2025_2026.nc


In [ ]:
# Kiểm tra nhanh output sau khi tải; không chuẩn hóa dữ liệu ở bước này.
from netCDF4 import Dataset, num2date

if not VN_FILE.exists():
    raise FileNotFoundError(f"Chưa có file: {VN_FILE}")

with Dataset(VN_FILE, "r") as dataset:
    time_var = dataset.variables["time"]
    dates = pd.DatetimeIndex(pd.to_datetime(num2date(time_var[:], time_var.units, calendar=getattr(time_var, "calendar", "standard"), only_use_cftime_datetimes=False)))
    rainfall = dataset.variables["rainfall"]
    print(f"File: {VN_FILE}")
    print(f"Shape (time, lat, lon): {rainfall.shape}")
    print(f"Date range: {dates.min()} -> {dates.max()}")
    print(f"Expected half-hour frames: {len(pd.date_range(START, pd.Timestamp(END) - pd.Timedelta(minutes=30), freq="30min")):,}")
    print(f"Actual frames: {len(dates):,}")
    print(f"Duplicate timestamps: {dates.duplicated().sum():,}")
    print(f"Missing timestamps: {len(pd.date_range(START, pd.Timestamp(END) - pd.Timedelta(minutes=30), freq="30min").difference(dates)):,}")

File: /content/drive/MyDrive/Rainfall_Nowcasting/IMERG/data/imerg_vietnam/imerg_vietnam_2025_2026.nc
Shape (time, lat, lon): (13104, 165, 80)
Date range: 2025-01-01 00:00:00 -> 2025-09-30 23:30:00
Expected half-hour frames: 35,040
Actual frames: 13,104
Duplicate timestamps: 0
Missing timestamps: 21,936
